In [3]:
import sys
print(sys.executable)

from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]")
         .appName("rfm")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

print(spark.version)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\python.exe
4.2.0


In [30]:
files = {
    "article": "../data/raw/article.parquet",
    "customer": "../data/raw/customer.parquet",
    "transactions": "../data/processed/transactions.parquet"
}

for name, path in files.items():
    print(f"\n========== {name.upper()} ==========")

    df = spark.read.parquet(path)

    df.printSchema()
    print(name, df.count())


========== ARTICLE ==========
root
 |-- article_id: long (nullable = true)
 |-- product_code: long (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: long (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: long (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: long (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: long (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: long (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: long (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: long (nullable = true)
 |-- index_group_name: string (nullable = t

In [ ]:
# tổng quan 3 bảng
from pyspark.sql import functions as F

dataframes = {
    "ARTICLE": article,
    "CUSTOMER": customer,
    "TRANSACTIONS": transactions
}

for name, df in dataframes.items():
    print(f"\n========== {name} ==========")
    print("Số dòng:", df.count())
    print("Số cột:", len(df.columns))
    df.printSchema()


========== ARTICLE ==========
Số dòng: 105542
Số cột: 25
root
 |-- article_id: long (nullable = true)
 |-- product_code: long (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: long (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: long (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: long (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: long (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: long (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: long (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: long (nullable = true)
 |-- index_group

In [ ]:
# kiểm tra thời gian giao dịch và giá cả
transactions.select(
    F.min("t_dat").alias("ngay_dau"),
    F.max("t_dat").alias("ngay_cuoi"),
    F.min("price").alias("gia_thap_nhat"),
    F.max("price").alias("gia_cao_nhat"),
    F.avg("price").alias("gia_trung_binh")
).show(truncate=False)

+-------------------+-------------------+--------------------+------------------+-------------------+
|ngay_dau           |ngay_cuoi          |gia_thap_nhat       |gia_cao_nhat      |gia_trung_binh     |
+-------------------+-------------------+--------------------+------------------+-------------------+
|2019-09-07 00:00:00|2020-09-22 00:00:00|1.694915254237288E-5|0.5067796610169492|0.02838354704124242|
+-------------------+-------------------+--------------------+------------------+-------------------+



In [ ]:
# kiểm tra null và nan
for name, df in dataframes.items():
    print(f"\n========== {name} ==========")

    expressions = []

    for c, dtype in df.dtypes:
        if dtype in ["double", "float"]:
            expressions.append(
                F.sum(
                    F.when(F.isnan(F.col(c)), 1).otherwise(0)
                ).alias(c + "_NaN")
            )

        expressions.append(
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c + "_NULL")
        )

    df.select(expressions).show(truncate=False)


========== ARTICLE ==========
+---------------+-----------------+--------------+--------------------+----------------------+-----------------------+----------------------------+------------------------------+----------------------+----------------------+------------------------------+--------------------------------+-------------------------------+---------------------------------+------------------+--------------------+---------------+---------------+-------------------+---------------------+---------------+-----------------+---------------------+-----------------------+----------------+
|article_id_NULL|product_code_NULL|prod_name_NULL|product_type_no_NULL|product_type_name_NULL|product_group_name_NULL|graphical_appearance_no_NULL|graphical_appearance_name_NULL|colour_group_code_NULL|colour_group_name_NULL|perceived_colour_value_id_NULL|perceived_colour_value_name_NULL|perceived_colour_master_id_NULL|perceived_colour_master_name_NULL|department_no_NULL|department_name_NULL|index_cod

In [35]:
#  kiểm tra kháo và giá không hợp lệ
print("TRANSACTIONS thiếu customer_id:",
      transactions.filter(F.col("customer_id").isNull()).count())

print("TRANSACTIONS thiếu article_id:",
      transactions.filter(F.col("article_id").isNull()).count())

print("TRANSACTIONS thiếu t_dat:",
      transactions.filter(F.col("t_dat").isNull()).count())

print("TRANSACTIONS thiếu price:",
      transactions.filter(F.col("price").isNull()).count())

print("PRICE <= 0:",
      transactions.filter(F.col("price") <= 0).count())

TRANSACTIONS thiếu customer_id: 0
TRANSACTIONS thiếu article_id: 0
TRANSACTIONS thiếu t_dat: 0
TRANSACTIONS thiếu price: 0
PRICE <= 0: 0


In [ ]:
# kiểm tra giao dịch không tìm thấy khách hàng hoặc sản phẩm
missing_customer = transactions.join(
    customer.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

missing_article = transactions.join(
    article.select("article_id"),
    on="article_id",
    how="left_anti"
)

print("Giao dịch không tìm thấy khách hàng:", missing_customer.count())
print("Giao dịch không tìm thấy sản phẩm:", missing_article.count())

Giao dịch không tìm thấy khách hàng: 0
Giao dịch không tìm thấy sản phẩm: 0
